# DSCI 552 - Homework 3

Name: Austin Oliver
<br>
Github Username: awoliver335
<br>
USC ID: 1949137702
<br>
Date: 10/10/2026

This notebook covers time-series feature extraction and ISLR Exercise 3.7.4.
I use the AReM files in `data/`. Run the cells in order from the repository
folder with Python 3, NumPy, pandas, and Jupyter installed.
The bootstrap uses 10,000 resamples and a random seed of 552.

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Imports and dataset settings:

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

data_dir = Path('data')
seed = 552
activities = ['bending1', 'bending2', 'cycling', 'lying',
              'sitting', 'standing', 'walking']
sensor_columns = ['avg_rss12', 'var_rss12', 'avg_rss13',
                  'var_rss13', 'avg_rss23', 'var_rss23']
expected_times = np.arange(0, 120000, 250)

#### Loading the AReM data

I use the [UCI AReM dataset](https://doi.org/10.24432/C5SS33), which contains
88 recordings across seven activities. Each recording is one instance with
six sensor series. The time column is excluded from the features.

A few files need minor formatting fixes: `bending2/dataset4.csv` uses spaces
instead of commas, and `cycling/dataset9.csv` and `cycling/dataset14.csv`
contain trailing commas. The loader handles these without changing the files.

`sitting/dataset8.csv` has 479 samples because timestamp 13,500 ms is missing.
I use the available samples rather than interpolate a value. The other
recordings have 480 samples. Zero measurements are kept.

In [2]:
def load_recording(path):
    """Read a recording, allowing for spaces and trailing commas."""
    rows = []
    for line in path.read_text(encoding='utf-8-sig').splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue

        values = line.rstrip(',').replace(',', ' ').split()
        if len(values) != 7:
            raise ValueError(f'Expected seven columns in {path}')
        rows.append([float(value) for value in values])

    frame = pd.DataFrame(rows, columns=['time', *sensor_columns])
    assert np.isfinite(frame.to_numpy()).all()
    assert frame['time'].is_unique and frame['time'].is_monotonic_increasing
    assert frame['time'].isin(expected_times).all()
    return frame


files = sorted(
    data_dir.glob('*/dataset*.csv'),
    key=lambda path: (path.parent.name, int(path.stem.replace('dataset', ''))),
)
records = {}
metadata_rows = []
for instance, path in enumerate(files, start=1):
    recording = load_recording(path)
    records[instance] = recording
    metadata_rows.append({
        'instance': instance,
        'activity': path.parent.name,
        'dataset_id': int(path.stem.replace('dataset', '')),
        'filename': path.name,
        'source_file': path.relative_to(data_dir).as_posix(),
        'n_samples': len(recording),
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
    })

metadata = pd.DataFrame(metadata_rows).set_index('instance')
activity_counts = metadata.groupby('activity').size().reindex(activities)
assert activity_counts.tolist() == [7, 6, 15, 15, 15, 15, 15]
assert metadata['n_samples'].value_counts().to_dict() == {480: 87, 479: 1}

print(f'Loaded {len(records)} recordings.')
display(activity_counts.rename('recordings').to_frame())
print('Recording with a missing sample:')
display(metadata.loc[metadata['n_samples'] != 480, ['source_file', 'n_samples']])
display(pd.DataFrame({'series': range(1, 7), 'sensor': sensor_columns}))

Loaded 88 recordings.


,recordings
activity,
bending1,7
bending2,6
cycling,15
lying,15
sitting,15
standing,15
walking,15


Recording with a missing sample:


,source_file,n_samples
instance,,
51,sitting/dataset8.csv,479


,series,sensor
0,1,avg_rss12
1,2,var_rss12
2,3,avg_rss13
3,4,var_rss13
4,5,avg_rss23
5,6,var_rss23


### (b) Train/test split

Following the assignment, I put datasets 1 and 2 from each bending activity
in the test set, along with datasets 1, 2, and 3 from every other activity.
The remaining files form the training set.

In [3]:
is_bending = metadata['activity'].isin(['bending1', 'bending2'])
test_limit = np.where(is_bending, 2, 3)
metadata['split'] = np.where(metadata['dataset_id'] <= test_limit, 'test', 'train')
train_ids = metadata.index[metadata['split'] == 'train']
test_ids = metadata.index[metadata['split'] == 'test']

split_counts = pd.crosstab(metadata['activity'], metadata['split']).reindex(
    index=activities, columns=['train', 'test']
)
split_counts['total'] = split_counts.sum(axis=1)
split_counts.loc['Total'] = split_counts.sum(axis=0)
display(split_counts)

# Compare file hashes to check for repeated recordings.
duplicate_rows = []
for _, group in metadata.groupby('sha256', sort=False):
    if len(group) > 1:
        duplicate_rows.append({
            'files': '; '.join(group['source_file']),
            'splits': '; '.join(group['split']),
            'crosses_train_test': group['split'].nunique() > 1,
        })
duplicates = pd.DataFrame(duplicate_rows)
print('Duplicate recordings:')
with pd.option_context('display.max_colwidth', None):
    display(duplicates)

split,train,test,total
activity,,,
bending1,5,2,7
bending2,4,2,6
cycling,12,3,15
lying,12,3,15
sitting,12,3,15
standing,12,3,15
walking,12,3,15
Total,69,19,88


Duplicate recordings:


,files,splits,crosses_train_test
0,cycling/dataset1.csv; cycling/dataset15.csv,test; train,True
1,lying/dataset1.csv; lying/dataset15.csv,test; train,True
2,lying/dataset3.csv; lying/dataset11.csv,test; train,True
3,lying/dataset5.csv; lying/dataset12.csv,train; train,False
4,lying/dataset7.csv; lying/dataset13.csv,train; train,False
5,lying/dataset9.csv; lying/dataset14.csv,train; train,False


The split gives 69 training recordings and 19 test recordings.
There are also six pairs of identical files in the data. Three pairs cross
the train/test split: cycling 1/15, lying 1/15, and lying 3/11.
I keep the assigned split, but these duplicates could make later test
performance look better than it would on new recordings.

### (c) Feature Extraction

#### i. Common time-domain features

Time-domain features describe a signal using its values and their order
in time. Common examples include:

- Mean and median for the typical signal level.
- Minimum, maximum, standard deviation, and quartiles for spread.
- Skewness and kurtosis for the shape of the distribution.
- Root mean square and energy for signal magnitude.
- Autocorrelation and the number of peaks for temporal behavior.

The [tsfresh feature reference](https://tsfresh.readthedocs.io/en/stable/text/list_of_features.html)
gives examples of these features. For this assignment, I extract the seven
requested statistics from each sensor series.

#### ii. Extracting the features

For each recording, I calculate the minimum, maximum, mean, median,
standard deviation, first quartile, and third quartile of all six series.
This gives $6 \times 7 = 42$ features per recording. For example, `mean_1`
is the mean of the first series, and `q3_6` is the third quartile of the sixth.

I use the sample standard deviation (`ddof=1`):

$$s = \sqrt{\frac{1}{m-1}\sum_{t=1}^{m}(x_t-\bar{x})^2}.$$

Quartiles use linear interpolation. The features stay in their original
units, and the displayed table is rounded to four decimal places.

In [4]:
stat_names = ['min', 'max', 'mean', 'median', 'std', 'q1', 'q3']
feature_columns = [f'{stat}_{series}' for series in range(1, 7) for stat in stat_names]


def extract_features(recording):
    result = {}
    for series, sensor in enumerate(sensor_columns, start=1):
        values = recording[sensor]
        result.update({
            f'min_{series}': values.min(),
            f'max_{series}': values.max(),
            f'mean_{series}': values.mean(),
            f'median_{series}': values.median(),
            f'std_{series}': values.std(ddof=1),
            f'q1_{series}': values.quantile(0.25, interpolation='linear'),
            f'q3_{series}': values.quantile(0.75, interpolation='linear'),
        })
    return result


features = pd.DataFrame.from_dict(
    {instance: extract_features(recording) for instance, recording in records.items()},
    orient='index',
).reindex(index=metadata.index, columns=feature_columns)
features.index.name = 'instance'

X_train = features.loc[train_ids].copy()
X_test = features.loc[test_ids].copy()
y_train = metadata.loc[train_ids, 'activity'].copy()
y_test = metadata.loc[test_ids, 'activity'].copy()

print(f'Feature matrix: {features.shape}')
print(f'Training: {X_train.shape}, test: {X_test.shape}')
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.float_format', '{:.4f}'.format):
    display(features)

Feature matrix: (88, 42)
Training: (69, 42), test: (19, 42)


,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1,min_2,max_2,mean_2,median_2,std_2,q1_2,q3_2,min_3,max_3,mean_3,median_3,std_3,q1_3,q3_3,min_4,max_4,mean_4,median_4,std_4,q1_4,q3_4,min_5,max_5,mean_5,median_5,std_5,q1_5,q3_5,min_6,max_6,mean_6,median_6,std_6,q1_6,q3_6
instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,37.2500,45.0000,40.6248,40.5000,1.4770,39.2500,42.0000,0.0000,1.3000,0.3586,0.4300,0.3226,0.0000,0.5000,4.0000,29.5000,19.0409,19.2500,4.4630,16.0000,23.2500,0.0000,7.2300,0.8325,0.5000,0.9657,0.0000,1.1200,27.2500,38.2500,34.3113,35.0000,2.1884,33.0000,36.0000,0.0000,1.9200,0.5706,0.4300,0.5829,0.0000,1.3000
2,38.0000,45.6700,42.8128,42.5000,1.4355,42.0000,43.6700,0.0000,1.2200,0.3724,0.4700,0.2892,0.0000,0.5000,2.0000,29.5000,20.0970,21.0000,3.8937,19.0000,22.2500,0.0000,5.7600,0.8764,0.5000,1.0473,0.0000,1.1450,27.6700,38.5000,33.0246,33.0000,1.9953,32.0000,34.5000,0.0000,3.1100,0.5711,0.4300,0.6010,0.0000,1.3000
3,35.0000,47.4000,43.9545,44.3300,1.5588,43.0000,45.0000,0.0000,1.7000,0.4263,0.4700,0.3387,0.0000,0.5000,6.5000,29.7500,22.1224,23.0000,3.0309,19.7500,24.0000,0.0000,4.4400,0.4973,0.4300,0.5507,0.0000,0.8300,29.0000,38.5000,35.5885,36.0000,1.9996,35.3625,36.5000,0.0000,1.7900,0.4933,0.4300,0.5135,0.0000,0.9400
4,33.0000,47.7500,42.1798,43.5000,3.6707,39.1500,45.0000,0.0000,3.0000,0.6960,0.5000,0.6309,0.0000,1.1200,8.5000,30.0000,22.1836,23.0000,3.8105,20.5000,24.3725,0.0000,5.1500,0.9899,0.8300,0.9537,0.4300,1.3000,20.0000,38.6700,33.4939,35.0000,3.8494,30.4575,36.3300,0.0000,2.1800,0.6135,0.5000,0.5243,0.0000,1.0000
5,33.0000,45.7500,41.6781,41.7500,2.2435,41.3300,42.7500,0.0000,2.8300,0.5360,0.5000,0.4055,0.4300,0.7100,3.0000,28.2500,19.0066,19.1250,4.0871,16.5000,22.0625,0.0000,6.4200,0.8419,0.5000,0.9288,0.4300,1.1200,23.6700,37.5000,29.8571,30.0000,2.4110,28.4575,31.2500,0.0000,1.7900,0.3833,0.4300,0.3892,0.0000,0.5000
6,37.0000,48.0000,43.4550,43.2500,1.3861,42.5000,45.0000,0.0000,1.5800,0.3781,0.4700,0.3156,0.0000,0.5000,5.7500,27.0000,15.7933,15.0000,3.8476,13.0000,18.2700,0.0000,10.0300,0.8494,0.5000,0.9958,0.4300,1.1200,8.0000,33.5000,23.0348,23.5000,2.4889,22.2500,24.0000,0.0000,5.2600,0.6796,0.5000,0.6225,0.4300,0.8700
7,36.2500,48.0000,43.9691,44.5000,1.6184,43.3100,44.6700,0.0000,1.5000,0.4131,0.4700,0.2631,0.4300,0.5000,1.5000,26.3300,15.8680,16.2500,3.7424,14.2500,18.0000,0.0000,5.1700,0.6664,0.4700,0.7890,0.0000,0.9400,11.3300,30.7500,22.1038,21.6700,3.3183,20.5000,23.7500,0.0000,2.9600,0.5553,0.4900,0.4878,0.0000,0.8300
8,12.7500,51.0000,24.5630,24.2500,3.7375,23.1875,26.5000,0.0000,6.8700,0.5908,0.4300,0.8374,0.0000,0.7100,0.0000,25.3300,19.1213,20.2500,4.2996,18.0000,22.0000,0.0000,6.7600,0.7368,0.4700,0.9270,0.0000,0.8700,10.6700,30.0000,23.4930,23.7500,3.6938,20.5000,27.0000,0.0000,4.9700,0.7002,0.5000,0.6937,0.4300,0.8700
9,0.0000,42.7500,27.4646,28.0000,3.5836,25.5000,30.0000,0.0000,7.7600,0.4497,0.4300,0.7672,0.0000,0.5000,7.5000,35.0000,20.8425,20.7500,3.8263,18.0000,22.5000,0.0000,5.7600,0.7799,0.5000,0.9044,0.0000,0.9400,0.0000,33.0000,17.6179,18.0000,5.0536,15.0000,20.7500,0.0000,6.7600,1.1221,0.8300,1.0123,0.4700,1.3000


#### iii. Standard deviations and bootstrap confidence intervals

Here I calculate the standard deviation of each feature across the 88
recordings. This differs from features such as `std_1`, which describe
variation within one recording.

$$\hat{\sigma}_j = \sqrt{\frac{1}{87}\sum_{i=1}^{88}(f_{ij}-\bar{f}_j)^2}.$$

To estimate a 90% confidence interval, I sample 88 rows with replacement
and recalculate all 42 standard deviations. I repeat this 10,000 times
and use the 5th and 95th percentiles as the interval endpoints. Sampling
whole rows keeps the features from each recording together.

I use all 88 instances for this part, as requested. The bootstrap assumes
independent, representative recordings; the duplicates and shared actor
limit how far these intervals can generalize.

In [5]:
def bootstrap_feature_std(values, n_resamples=10_000, seed=seed, batch_size=250):
    """Resample recordings and calculate the SD of each feature."""
    n_recordings, n_features = values.shape
    rng = np.random.default_rng(seed)
    estimates = np.empty((n_resamples, n_features))

    # Use small batches to avoid keeping all resampled data in memory.
    for start in range(0, n_resamples, batch_size):
        stop = min(start + batch_size, n_resamples)
        indices = rng.integers(0, n_recordings, size=(stop - start, n_recordings))
        estimates[start:stop] = values[indices].std(axis=1, ddof=1)
    return estimates


bootstrap_std = bootstrap_feature_std(features.to_numpy())
lower, upper = np.quantile(bootstrap_std, [0.05, 0.95], axis=0, method='linear')
std_intervals = pd.DataFrame({
    'sample_std': features.std(ddof=1),
    'ci_90_lower': lower,
    'ci_90_upper': upper,
})
std_intervals.index.name = 'feature'

print('Feature standard deviations and 90% bootstrap confidence intervals:')
with pd.option_context('display.max_rows', None, 'display.float_format', '{:.6f}'.format):
    display(std_intervals)

Feature standard deviations and 90% bootstrap confidence intervals:


,sample_std,ci_90_lower,ci_90_upper
feature,,,
min_1,9.569975,8.262167,10.761378
max_1,4.394362,3.328017,5.284915
mean_1,5.335718,4.704031,5.886357
median_1,5.440054,4.790278,6.004313
std_1,1.772153,1.572160,1.950031
q1_1,6.153590,5.581620,6.652410
q3_1,5.138925,4.331936,5.840872
min_2,0.000000,0.000000,0.000000
max_2,5.062729,4.622848,5.402171


These intervals estimate uncertainty in each feature's standard deviation
across recordings. A larger standard deviation does not necessarily make
a feature more useful for classification, especially when the features
have different scales.

#### iv. Choosing three feature types

I would start with **mean, maximum, and standard deviation**. The mean
captures the typical sensor level, the maximum captures peaks, and the
standard deviation measures how much the signal varies during an activity.
Together, they describe different aspects of a recording.

Using these three statistics for all six series gives 18 features.
This is a starting choice based on their interpretation; I would need
validation results to judge their predictive value. Maximum is sensitive
to outliers, so a quartile would be worth comparing as an alternative.

In [6]:
selected_types = ['mean', 'max', 'std']
selected_columns = [f'{stat}_{series}' for series in range(1, 7) for stat in selected_types]
X_train_selected = X_train[selected_columns].copy()
X_test_selected = X_test[selected_columns].copy()

print(f'Selected features: training {X_train_selected.shape}, test {X_test_selected.shape}')
display(pd.DataFrame({
    'series': range(1, 7),
    'sensor': sensor_columns,
    'features': [', '.join(f'{stat}_{series}' for stat in selected_types)
                 for series in range(1, 7)],
}))

Selected features: training (69, 18), test (19, 18)


,series,sensor,features
0,1,avg_rss12,"mean_1, max_1, std_1"
1,2,var_rss12,"mean_2, max_2, std_2"
2,3,avg_rss13,"mean_3, max_3, std_3"
3,4,var_rss13,"mean_4, max_4, std_4"
4,5,avg_rss23,"mean_5, max_5, std_5"
5,6,var_rss23,"mean_6, max_6, std_6"


A few final checks on the feature matrices and the confidence intervals:

In [7]:
assert features.shape == (88, 42)
assert X_train.shape == (69, 42) and X_test.shape == (19, 42)
assert set(train_ids).isdisjoint(test_ids)
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert X_train_selected.shape == (69, 18) and X_test_selected.shape == (19, 18)
assert np.isfinite(features.to_numpy()).all()
assert std_intervals.shape == (42, 3)
assert np.isfinite(std_intervals.to_numpy()).all()
assert (std_intervals['ci_90_lower'] >= 0).all()
assert (std_intervals['ci_90_lower'] <= std_intervals['ci_90_upper']).all()

print('All checks passed.')

All checks passed.


## 2. ISLR 3.7.4

Reference: James, Witten, Hastie, and Tibshirani,
[*An Introduction to Statistical Learning*](https://www.statlearning.com/s/ISLRSeventhPrinting.pdf),
Section 3.7, Exercise 4 (pp. 120-121).

Both models use ordinary least squares on the same 100 observations.
The cubic model includes the linear model as the special case where the
coefficients of $X^2$ and $X^3$ are zero.

### (a) Linear relationship: training RSS

The cubic model has training RSS less than or equal to the linear model:

$$\mathrm{RSS}_{\text{train,cubic}} \leq \mathrm{RSS}_{\text{train,linear}}.$$

It can reproduce any fitted line, and the extra terms may fit some of the
training noise. Adding those terms cannot increase the minimum training
RSS, even when the true relationship is linear. Equality is possible if
the extra terms do not improve the fit.

### (b) Linear relationship: test RSS

I would expect the linear model to have lower test RSS. Since the true
relationship is linear, the extra terms do not reduce approximation bias
but add variance from estimating more coefficients. This makes the cubic
model more likely to fit training noise. The linear model should do better
on average, although a particular test set could give the opposite result.

### (c) Nonlinear relationship: training RSS

The cubic model still has training RSS less than or equal to the linear
model. The reasoning from part (a) depends on the models being nested,
so it also applies when the true relationship is nonlinear. The extra
terms may now capture real curvature as well as noise.

### (d) Nonlinear relationship: test RSS

There is not enough information to say which model will have lower test
RSS. If the cubic terms capture substantial curvature, their reduction
in bias may outweigh the added variance. If the relationship is close to
linear or the data are noisy, the linear model may still do better.
I would compare them using a validation set or cross-validation.